# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.  
LLMs are great at this!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Re-writing</h2>
            <span style="color:#181;">LLMs have made it simple to do something that was considered impossible only a few years ago.
            This approach can be applied to almost any business vertical, and it's similar to the advanced techniques
            we used on Week 5.</span>
        </td>
    </tr>
</table>

In [14]:
# LiteLLM gives one common completion() interface across providers such as OpenAI, Groq, and Ollama.
from litellm import completion
from dotenv import load_dotenv
import json
import subprocess

# Batch handles large-scale hosted preprocessing jobs; Item represents one scraped product record.
# from pricer.batch import Batch
from pricer.batch_local import Batch
from pricer.items import Item

# Load API keys and other local settings from .env, replacing any stale values already in the session.
load_dotenv(override=True)


True

In [19]:
import importlib, pricer.batch_local
importlib.reload(pricer.batch_local)
from pricer.batch_local import Batch
print(pricer.batch_local.__file__) 

c:\Users\HP\Desktop\LLM_ENGINEERING\week6\pricer\batch_local.py


# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

## For this lab

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [15]:
# Toggle between the small teaching dataset and the full production-sized dataset.
# Lite mode is cheaper and faster; full mode gives the model much more product coverage.
LITE_MODE = False


In [16]:
# Pick the raw dataset variant that matches the mode above.
username = "amaima"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

# Load the Hugging Face splits, then combine them temporarily so preprocessing can run uniformly.
train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Loaded {len(items):,} items")

print(items[0])


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: a2e9ca49-279b-4f87-bfb2-bd8615d37ef6)')' thrown while requesting HEAD https://huggingface.co/datasets/amaima/items_raw_full/resolve/main/README.md
Retrying in 1s [Retry 1/5].


Loaded 820,000 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item We

In [6]:
items[2].id

2

In [17]:
# Give every item a stable numeric id before batching.
# The id becomes the batch custom_id, so returned summaries can be matched back to the right product.
for index, item in enumerate(items):
    item.id = index


In [7]:
# The system prompt defines the normalized product format we want from the LLM.
# Keeping this structure consistent makes the rewritten descriptions easier to train on later.
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""


In [5]:
SYSTEM_PROMPT = """Create a concise product description. You MUST strictly follow the format. Any deviation is incorrect. Do not include part numbers.

Follow EXACT format (no exceptions):
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features

Rules:
- Output ONLY these 5 lines in this exact order
- Do NOT add extra lines, text, punctuation, or explanations
- Do NOT rename, reorder, or modify fields
- Do NOT include part/model numbers
- Each field must be minimal and precise"""

In [7]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [9]:
print(items[1].full)

KiCA JETFAN 1.0 Mini Electric Air Duster Fan Blower Aluminum, 86,000RPM up to 11m/s Wind Speed for Computer Keyboard Electronics Cleaning, Outdoor Hiking, Camping, Air Cushion Inflation, BBQ
['【Super Strong Wind】The 9.99Wh motor can reach up to 100,000 rpm, and the fastest speed is 11.5 meters per second blast.', '【Mini & Portable】Come with a mini and portable body,smaller than your phone.', '【Splendid Craftsmanship】The delicate metal workmanship makes the turbofan always be eye-catching. Plaything or handicraft, switching between each time you use.', '【Keep You Away From Heat】In hot summer, the mini fan in your hand is not cool enough? The turbofan can be adjustable in 4 levels. You can even switch to the 11.5 meters per second wind if you are not satisfied with the breeze.', '【Multi-Functional】KiCA Jetfan air duster can be used for cleaning computer, laptop, keyboard, instruments and camera, drying hair, shooting equipment and astronomical equipment, inflating balloons and outdoor in

In [10]:
print(items[-1].full)

DuraGo BP1082 C Rear Ceramic Brake Pad
['DuraGo offers a comprehensive Brake Friction program for every budget and consumer. Everyday driving safety and reliability were of paramount importance in developing our Brake Pad program. All brake friction formulas have been independently tested in North American for dyno performance, wear and SAE 2521 noise squeal protocols. Our ceramic material formulation adds stability and predictability to the brake pad. Integrally molded with superior bond retention and shear strength for optimum braking performance and extremely quiet operation. DuraGo Brake Pads provide superior vehicle control, driver comfort and safety over a wide range of driving conditions - maximizing driver confidence.']
['Precise chamfers ensuring the maximum friction area with the most stopping power and quiet operation', 'Precision cut backing plates reduce vibration in the brake caliper', 'Noise reducing shims dampen brake pad vibration', 'Slotted for exceptional noise and v

In [12]:
# Try one single product first before launching a batch job.
# This checks that the prompt, model, credentials, and cost reporting all behave as expected.
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[6].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


Title: Karakuri Creation Group Egg Puzzle Box  
Category: Home & Garden / Décor  
Brand: Karakuri Creation Group  
Description: A handcrafted cherry‑wood egg puzzle box created by Akio Kamei in Japan.  
Details: Dimensions 60 mm × 60 mm × 80 mm, weight 12.6 oz, designed for ages 6+ and adults, featuring elegant mechanical design and precise wooden craftsmanship.

Input tokens: 291
Output tokens: 120
Cost: 0.006 cents


In [13]:

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[6].full}]
response = completion(messages=messages, model="ollama/qwen2.5-coder:7b", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


Title: Karakuri Creation Group Egg Puzzle Box

Category: Toys & Games

Brand: Karakuri Creation Group

Description: A beautifully crafted, hand-carved wooden egg puzzle box by Akio Kamei in Japan.

Details: Handmade from the finest cherrywood, this intricate puzzle box is a perfect gift for those who appreciate fine craftsmanship and enjoy engaging with beautiful toys.

Input tokens: 259
Output tokens: 79
Cost: 0.000 cents


In [8]:

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[6].full}]
response = completion(messages=messages, model="ollama/llama3.2:latest", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


Title: Handmade Cherry Wood Puzzle Box
Category: Home Decor
Brand: Karakuri Creation Group
Description: A beautifully crafted, Karakuri puzzle box made from the finest cherrywood.
Details: Features intricate designs and no force is necessary.

Input tokens: 318
Output tokens: 52
Cost: 0.000 cents


In [23]:
# Model name used inside each batch request body.
# Groq's batch API expects OpenAI-compatible model naming here.
MODEL = "openai/gpt-oss-20b"


In [24]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [20]:
items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [21]:
items[-1]

<DuraGo BP1082 C Rear Ceramic Brake Pad = $29.81>

In [25]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "openai/gpt-oss-20b", "messages": [{"role": "system", "content": "Create a concise product description. You MUST strictly follow the format. Any deviation is incorrect. Do not include part numbers.\\n\\nFollow EXACT format (no exceptions):\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features\\n\\nRules:\\n- Output ONLY these 5 lines in this exact order\\n- Do NOT add extra lines, text, punctuation, or explanations\\n- Do NOT rename, reorder, or modify fields\\n- Do NOT include part/model numbers\\n- Each field must be minimal and precise"}, {"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of

In [26]:
def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [27]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [28]:
import os
from groq import Groq

# Create the Groq client from the API key loaded earlier by load_dotenv().
groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))


In [ ]:
# Hosted Groq batch upload kept for reference. This needs a paid/eligible Groq plan.
with open("jsonl/0_1000.jsonl", "rb") as f:
    response = groq.files.create(file=f, purpose="batch")
response

In [ ]:
# Hosted Groq batch file id kept for reference:
file_id = response.id
# file_idb

In [ ]:
# Hosted Groq batch creation kept for reference:
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
# response

In [ ]:
# Hosted Groq batch status check kept for reference:
result = groq.batches.retrieve(response.id)
# result


In [ ]:
# Hosted Groq result download kept for reference:
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [ ]:
# Read the completed batch output and attach each rewritten summary to its original Item.
with open("jsonl/batch_results.jsonl", "r") as f:
    for line in f:
        json_line = json.loads(line)
        if json_line.get("error"):
            print(f"Skipping item {json_line['custom_id']} because local generation failed: {json_line['error']}")
            continue

        # custom_id is the item id assigned earlier, so it points back into the combined items list.
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


In [ ]:
print(items[0].full)

In [ ]:
print(items[1000].summary)

## I've put exactly this logic into a Batch class

- Divides items into groups of 1,000
- Kicks off batches for each
- Allows us to monitor and collect the results when complete

## COSTS

Using Groq, for me - this cost under $1 for the Lite dataset and under $30 for the big dataset

But you don't need to pay anything! In the next lab, you can load my pre-processed results

In [20]:
# Create provider batch files in chunks of 1,000 items.
Batch.create(items, LITE_MODE)


Created 820 batches


In [ ]:
# Submit any prepared batch files that have not already been sent.
Batch.run()


  0%|          | 0/820 [00:00<?, ?it/s]

  8%|▊         | 80/1000 [1:21:45<15:40:07, 61.31s/it]


In [ ]:
# Download completed batch results and merge the generated summaries back into the item objects.
Batch.fetch()


In [ ]:
# Sanity-check that every product received a summary before publishing the dataset.
for index, item in enumerate(items):
    if not item.summary:
        print(index)


In [ ]:
print(items[10234].summary)

In [ ]:
# Remove fields that are useful only during preprocessing.
# The hub dataset should keep the clean training signal, not bulky raw text or temporary ids.
for item in items:
    item.full = None
    item.id = None


## Push the final dataset to the hub

If lite mode, we'll only push the lite dataset

If full mode, we'll push both datasets (in case you decide to use lite later)

In [ ]:
# Push the processed dataset under the final Hugging Face repository names.
username = "amaima"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    # Lite mode publishes only the small split used for quick experiments.
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    # Full mode publishes the complete dataset and also derives a lite subset for convenience.
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)


## And here they are!

https://huggingface.co/datasets/ed-donner/items_lite

https://huggingface.co/datasets/ed-donner/items_full
